In [1]:
import os
import numpy as np
import pandas as pd

# Paths: notebook is in zavala_electricity_market, data in dataset/IM-3-GO-WEST
# Run from project root (prj_market) or from zavala_electricity_market
_cwd = os.getcwd()
if os.path.basename(_cwd) == "zavala_electricity_market":
    BASE_DIR = os.path.dirname(_cwd)
else:
    BASE_DIR = _cwd
DATA_DIR = os.path.join(BASE_DIR, "dataset", "IM-3-GO-WEST")
if not os.path.isdir(DATA_DIR):
    DATA_DIR = os.path.join(_cwd, "dataset", "IM-3-GO-WEST")

# Ensure zavala_electricity_market is on path (when running from project root)
import sys
ZAVALA_DIR = os.path.join(BASE_DIR, "zavala_electricity_market")
if os.path.isdir(ZAVALA_DIR) and ZAVALA_DIR not in sys.path:
    sys.path.insert(0, ZAVALA_DIR)

from zavala_funcs import (
    zavala,
    zavala_cvar,
    zavala_deterministic_da,
    zavala_rt_energy_only,
    expected_caps_from_scenarios,
    price_distortion,
    probability_feasible,
    expected_cumulative_regret,
    compute_social_surplus,
    tail_worst_indices_by_value,
    _stack_rt,
)

print("Data dir:", DATA_DIR)
print("Files:", os.listdir(DATA_DIR) if os.path.isdir(DATA_DIR) else "not found")

Data dir: /Users/hrithiknambiar/Desktop/Research/prj_market/dataset/IM-3-GO-WEST
Files: ['nodal_load.csv', 'nodal_wind.csv', 'thermal_gens.csv', 'nodal_solar.csv']


In [2]:
# Load nodal time series (rows = time, columns = bus_XXXXX)
solar = pd.read_csv(os.path.join(DATA_DIR, "nodal_solar.csv"))
wind = pd.read_csv(os.path.join(DATA_DIR, "nodal_wind.csv"))
load_df = pd.read_csv(os.path.join(DATA_DIR, "nodal_load.csv"))
thermal_df = pd.read_csv(os.path.join(DATA_DIR, "thermal_gens.csv"))

T = len(solar)
assert len(wind) == T and len(load_df) == T, "Solar, wind, load must have same length"
print(f"Time steps: {T}")
print(f"Solar columns: {solar.shape[1]}, Wind: {wind.shape[1]}, Load: {load_df.shape[1]}")
print(f"Thermal generators: {len(thermal_df)}")

Time steps: 8760
Solar columns: 125, Wind: 125, Load: 125
Thermal generators: 280


In [ ]:
# Choose buses with non-trivial solar: columns with max > threshold
solar_cols = [c for c in solar.columns if solar[c].max() > 50]
wind_cols = [c for c in wind.columns if wind[c].max() > 50]
# Pick 3 solar and 3 wind (unreliable)
num_solar, num_wind = 3, 3
solar_buses = solar_cols[:num_solar] if len(solar_cols) >= num_solar else list(solar.columns[:num_solar])
wind_buses = wind_cols[:num_wind] if len(wind_cols) >= num_wind else list(wind.columns[:num_wind])

# Reliable: aggregate thermal by bus, pick 4 buses with largest capacity
thermal_by_bus = thermal_df.groupby("Bus")["Max_Cap"].sum().sort_values(ascending=False)
thermal_buses_numeric = list(thermal_by_bus.head(4).index)  # e.g. [408441, 135041, ...]
thermal_bus_cols = [f"bus_{b}" for b in thermal_buses_numeric]  # for load alignment if needed

# Load: use total system load (sum over all buses)
load_total = load_df.sum(axis=1).values  # (T,)

print("Solar buses (unreliable):", solar_buses)
print("Wind buses (unreliable):", wind_buses)
print("Thermal buses (reliable):", thermal_buses_numeric)
print("Load: system total (sum over all buses)")

Solar buses (unreliable): ['bus_100931', 'bus_200261', 'bus_201361']
Wind buses (unreliable): ['bus_100931', 'bus_102281', 'bus_105701']
Thermal buses (reliable): [500991, 605141, 408441, 261001]
Load: system total (sum over all buses)


In [4]:
def build_real_data_instance(solar_df, wind_df, load_total_vec, thermal_by_bus, thermal_buses_numeric,
                              solar_buses, wind_buses, start_idx, num_scenarios, rng=None):
    """
    Build (probs, mc_g_i, mv_d_j, g_i_bar, d_j_bar) from real data for one time window.
    - start_idx: first time index
    - num_scenarios: number of consecutive time steps (scenarios)
    """
    if rng is None:
        rng = np.random.default_rng()
    end_idx = start_idx + num_scenarios
    S = num_scenarios

    # Scenario probabilities: near-uniform (same idea as s_real10_mix)
    kappa = 1500.0
    alpha = np.full(S, kappa / S)
    probs = rng.dirichlet(alpha)
    probs = probs / probs.sum()

    # Marginal costs: cheap for unreliable (solar/wind), higher for reliable (thermal)
    mc_unrel = rng.uniform(8.0, 14.0, size=6)
    mc_rel = rng.uniform(35.0, 55.0, size=4)
    mc_g_i = np.concatenate([mc_unrel, mc_rel]).astype(float)

    # Single inelastic load (VOLL)
    mv_d_j = np.array([1000.0], dtype=float)

    # Generator capacities per scenario (S x 10)
    # Columns 0..2: solar, 3..5: wind, 6..9: thermal (constant)
    solar_vals = solar_df.loc[start_idx:end_idx - 1, solar_buses].values  # (S, 3)
    wind_vals = wind_df.loc[start_idx:end_idx - 1, wind_buses].values    # (S, 3)
    unrel_caps = np.clip(np.hstack([solar_vals, wind_vals]), 0.0, None)  # (S, 6)

    rel_caps = np.array([thermal_by_bus[b] for b in thermal_buses_numeric], dtype=float)
    rel_caps = np.broadcast_to(rel_caps, (S, 4))  # (S, 4) constant across scenarios

    g_i_bar = np.hstack([unrel_caps, rel_caps])  # (S, 10)

    # Demand: system total load for each scenario (S x 1)
    d_j_bar = load_total_vec[start_idx:end_idx].reshape(-1, 1).astype(float)
    d_j_bar = np.clip(d_j_bar, 1e-6, None)  # avoid zeros

    return probs, mc_g_i, mv_d_j, g_i_bar, d_j_bar

In [5]:
# Quick sanity check: one small window
NUM_SCENARIOS = 500
rng = np.random.default_rng(42)
probs, mc_g_i, mv_d_j, g_i_bar, d_j_bar = build_real_data_instance(
    solar, wind, load_total, thermal_by_bus, thermal_buses_numeric,
    solar_buses, wind_buses, start_idx=0, num_scenarios=min(NUM_SCENARIOS, T), rng=rng
)
print("probs.shape:", probs.shape, "sum:", probs.sum())
print("g_i_bar.shape:", g_i_bar.shape)
print("d_j_bar.shape:", d_j_bar.shape)
print("Unreliable (solar+wind) sample mean:", g_i_bar[:, :6].mean(axis=0))
print("Reliable (thermal) constant:", g_i_bar[0, 6:])
print("Load sample:", d_j_bar[:5].ravel())

probs.shape: (500,) sum: 1.0
g_i_bar.shape: (500, 10)
d_j_bar.shape: (500, 1)
Unreliable (solar+wind) sample mean: [ 67.33335058  17.28        28.36311076 660.0882676  133.714
 215.59573142]
Reliable (thermal) constant: [6922.33 6837.62 6742.08 6521.69]
Load sample: [75073. 72985. 71226. 70105. 69772.]


In [6]:
def run_zavala_one_instance(probs, mc_g_i, mv_d_j, g_i_bar, d_j_bar):
    """Run stochastic, CVaR, and deterministic Zavala for one instance. Returns dict of metrics.
    Same logic as run_zavala.py, no changes to external files.
    """
    # ----- Stochastic Zavala -----
    z_g_i, z_d_j, Z_G, Z_D, z_pi, z_Pi = zavala(probs, mc_g_i, mv_d_j, g_i_bar, d_j_bar)
    prob_f = probability_feasible(probs, z_g_i, z_d_j, g_i_bar, d_j_bar)
    z_dist = price_distortion(probs, z_pi, z_Pi)
    z_reg = expected_cumulative_regret(probs, z_g_i, z_d_j, z_pi, mc_g_i, mv_d_j, g_i_bar, d_j_bar)
    ss_stoch = compute_social_surplus(probs, mc_g_i, mv_d_j, g_da=z_g_i, d_da=z_d_j, G_rt=Z_G, D_rt=Z_D)

    # ----- CVaR Zavala -----
    cvar_g_i, cvar_d_j, C_G, C_D, cvar_pi, cvar_Pi, _ = zavala_cvar(probs, mc_g_i, mv_d_j, g_i_bar, d_j_bar)
    cvar_dist = price_distortion(probs, cvar_pi, cvar_Pi)
    cvar_reg = expected_cumulative_regret(probs, cvar_g_i, cvar_d_j, cvar_pi, mc_g_i, mv_d_j, g_i_bar, d_j_bar)
    ss_cvar = compute_social_surplus(probs, mc_g_i, mv_d_j, g_da=cvar_g_i, d_da=cvar_d_j, G_rt=C_G, D_rt=C_D)

    # ----- Deterministic (expected capacities) -----
    gbar_det, dbar_det = expected_caps_from_scenarios(probs, g_i_bar, d_j_bar)
    g_det, d_det, pi_det = zavala_deterministic_da(mc_g_i, mv_d_j, gbar_det, dbar_det)
    G_det_list, D_det_list, Pi_det_list = [], [], []
    for p in range(len(probs)):
        Gp, Dp, Pi_p = zavala_rt_energy_only(mc_g_i, mv_d_j, g_det, d_det, g_i_bar[p], d_j_bar[p])
        G_det_list.append(Gp)
        D_det_list.append(Dp)
        Pi_det_list.append(Pi_p)
    G_det_rt, D_det_rt = _stack_rt(G_det_list, D_det_list)
    Pi_det = np.array(Pi_det_list)
    det_dist = price_distortion(probs, pi_det, Pi_det)
    det_reg = expected_cumulative_regret(probs, g_det, d_det, pi_det, mc_g_i, mv_d_j, g_i_bar, d_j_bar)
    ss_det = compute_social_surplus(probs, mc_g_i, mv_d_j, g_da=g_det, d_da=d_det, G_rt=G_det_rt, D_rt=D_det_rt)

    # ----- Tail metrics (5% worst by high neg-surplus) -----
    tail = 0.05
    stoch_tail_idx = tail_worst_indices_by_value(ss_stoch["ss_per_scenario"], probs, tail=tail, worst="high")
    cvar_tail_idx = tail_worst_indices_by_value(ss_cvar["ss_per_scenario"], probs, tail=tail, worst="high")
    det_tail_idx = tail_worst_indices_by_value(ss_det["ss_per_scenario"], probs, tail=tail, worst="high")

    stoch_tail_welfare = -np.mean(ss_stoch["ss_per_scenario"][stoch_tail_idx])
    cvar_tail_welfare = -np.mean(ss_cvar["ss_per_scenario"][cvar_tail_idx])
    det_tail_welfare = -np.mean(ss_det["ss_per_scenario"][det_tail_idx])

    stoch_tail_dist = np.mean(np.abs(z_pi - np.array(z_Pi)[stoch_tail_idx]))
    cvar_tail_dist = np.mean(np.abs(cvar_pi - np.array(cvar_Pi)[cvar_tail_idx]))
    det_tail_dist = np.mean(np.abs(pi_det - Pi_det[det_tail_idx]))

    return {
        "prob_feasible": prob_f,
        "stoch_distortion": z_dist, "stoch_regret": z_reg, "stoch_ss": ss_stoch["E_social_surplus"],
        "stoch_tail_welfare": stoch_tail_welfare, "stoch_tail_distortion": stoch_tail_dist,
        "cvar_distortion": cvar_dist, "cvar_regret": cvar_reg, "cvar_ss": ss_cvar["E_social_surplus"],
        "cvar_tail_welfare": cvar_tail_welfare, "cvar_tail_distortion": cvar_tail_dist,
        "det_distortion": det_dist, "det_regret": det_reg, "det_ss": ss_det["E_social_surplus"],
        "det_tail_welfare": det_tail_welfare, "det_tail_distortion": det_tail_dist,
    }

In [8]:
NUM_INSTANCES = 10
NUM_SCENARIOS = 500
rng = np.random.default_rng(2025)

max_start = T - NUM_SCENARIOS
if max_start <= 0:
    raise ValueError(f"Need at least {NUM_SCENARIOS} time steps; have {T}")

# Random start indices for each instance (non-overlapping or random)
start_indices = rng.integers(0, max_start + 1, size=NUM_INSTANCES)

results_list = []
for i in range(NUM_INSTANCES):
    start = int(start_indices[i])
    probs, mc_g_i, mv_d_j, g_i_bar, d_j_bar = build_real_data_instance(
        solar, wind, load_total, thermal_by_bus, thermal_buses_numeric,
        solar_buses, wind_buses, start_idx=start, num_scenarios=NUM_SCENARIOS, rng=rng
    )
    print(f"the shapes of probs, mc_g_i, mv_d_j, g_i_bar, d_j_bar are {probs.shape}, {mc_g_i.shape}, {mv_d_j.shape}, {g_i_bar.shape}, {d_j_bar.shape}")
    res = run_zavala_one_instance(probs, mc_g_i, mv_d_j, g_i_bar, d_j_bar)
    results_list.append(res)
    print(f"Instance {i+1}/{NUM_INSTANCES} (start={start}) done.")

the shapes of probs, mc_g_i, mv_d_j, g_i_bar, d_j_bar are (500,), (10,), (1,), (500, 10), (500, 1)


/Users/hrithiknambiar/Desktop/Research/prj_market/.venv/lib/python3.11/site-packages/cvxpy/problems/problem.py:167: UserWarning: Objective contains too many subexpressions. Consider vectorizing your CVXPY code to speed up compilation.
  warnings.warn("Objective contains too many subexpressions. "


SolverError: Solver 'GUROBI' failed. Try another solver, or solve with verbose=True for more information.

In [ ]:
# Aggregate and average
keys = list(results_list[0].keys())
means = {k: np.mean([r[k] for r in results_list]) for k in keys}
stds = {k: np.std([r[k] for r in results_list]) for k in keys}

print("============== Real-data results (averaged over {} instances) ================".format(NUM_INSTANCES))
print("Distortion (DA vs E[RT price]):")
print("  Stochastic:", means["stoch_distortion"], "±", stds["stoch_distortion"])
print("  CVaR:", means["cvar_distortion"], "±", stds["cvar_distortion"])
print("  Deterministic:", means["det_distortion"], "±", stds["det_distortion"])
print("E[Social Surplus]:")
print("  Stochastic:", means["stoch_ss"], "±", stds["stoch_ss"])
print("  CVaR:", means["cvar_ss"], "±", stds["cvar_ss"])
print("  Deterministic:", means["det_ss"], "±", stds["det_ss"])
print("Tail (5%) welfare (mean positive SS in worst tail):")
print("  Stochastic:", means["stoch_tail_welfare"], "±", stds["stoch_tail_welfare"])
print("  CVaR:", means["cvar_tail_welfare"], "±", stds["cvar_tail_welfare"])
print("  Deterministic:", means["det_tail_welfare"], "±", stds["det_tail_welfare"])
print("Tail (5%) price distortion:")
print("  Stochastic:", means["stoch_tail_distortion"], "±", stds["stoch_tail_distortion"])
print("  CVaR:", means["cvar_tail_distortion"], "±", stds["cvar_tail_distortion"])
print("  Deterministic:", means["det_tail_distortion"], "±", stds["det_tail_distortion"])
print("Probability feasible:", means["prob_feasible"], "±", stds["prob_feasible"])

In [ ]:
summary = pd.DataFrame({
    "Method": ["Stochastic", "CVaR", "Deterministic"] * 3,
    "Metric": ["Distortion", "Distortion", "Distortion", "E[SS]", "E[SS]", "E[SS]", "Tail welfare", "Tail welfare", "Tail welfare"],
    "Mean": [
        means["stoch_distortion"], means["cvar_distortion"], means["det_distortion"],
        means["stoch_ss"], means["cvar_ss"], means["det_ss"],
        means["stoch_tail_welfare"], means["cvar_tail_welfare"], means["det_tail_welfare"],
    ],
    "Std": [
        stds["stoch_distortion"], stds["cvar_distortion"], stds["det_distortion"],
        stds["stoch_ss"], stds["cvar_ss"], stds["det_ss"],
        stds["stoch_tail_welfare"], stds["cvar_tail_welfare"], stds["det_tail_welfare"],
    ],
})
display(summary)